In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [ ]:
%load_ext autoreload
%autoreload 2

# Constrained docking — technology demo

This notebook demonstrates **all** constrained-docking capabilities exposed by
the CLI (except `self_test`). The platform derives harmonic MCS constraints
server-side from a **reference ligand** identity and a required **reference
pose** SDF.

Prerequisites: `deeporigin login` and a registered `deeporigin.constrained-docking`
tool on your org.

## Setup

In [ ]:
from deeporigin.drug_discovery import (
    BRD_DATA_DIR,
    ConstrainedDocking,
    Docking,
    Ligand,
    LigandSet,
    Pocket,
    Protein,
)
from deeporigin.platform import DeepOriginClient

REMOTE_PREFIX = "notebooks/constrained-docking"

In [ ]:
client = DeepOriginClient.from_disk()
client

## Shared inputs

Upload the BRD protein, reference ligand, and analog test ligands once. The binding
pocket is loaded from the local test fixture (`tests/fixtures/files/pocketfinder/pocket_1.pdb`).

In [ ]:
protein = Protein.from_file(BRD_DATA_DIR / "brd.pdb")
protein.remove_water()
protein.sync(client=client, remote_path=f"{REMOTE_PREFIX}/brd.pdb")
protein.id

In [ ]:
from pathlib import Path

repo_root = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "pyproject.toml").is_file() and (p / "tests").is_dir()
)
pocket_fixture = repo_root / "tests/fixtures/files/pocketfinder/pocket_1.pdb"
pocket = Pocket.from_pdb_file(pocket_fixture, name="pocket_1")
pocket.box_size_x = pocket.box_size_y = pocket.box_size_z = 30.0
pocket.get_center()
pocket

In [ ]:
reference_ligand = Ligand.from_sdf(BRD_DATA_DIR / "brd-2.sdf")
reference_ligand.sync(
    client=client,
    remote_path=f"{REMOTE_PREFIX}/brd-2.sdf",
)
reference_ligand

In [ ]:
analog_paths = [BRD_DATA_DIR / "brd-3.sdf", BRD_DATA_DIR / "brd-4.sdf"]
analogs = LigandSet(
    ligands=[Ligand.from_sdf(path) for path in analog_paths],
)
for idx, lig in enumerate(analogs):
    lig.sync(
        client=client,
        remote_path=f"{REMOTE_PREFIX}/brd-{idx + 3}.sdf",
    )
analogs

## 1. Dock-then-constrain pipeline

Standard dock the reference ligand, upload the best pose as `reference_pose`, then
constrained-dock an analog.

In [ ]:
ref_docking = Docking(
    protein=protein,
    pocket=pocket,
    ligand=reference_ligand,
    client=client,
)
ref_poses = ref_docking.run()
reference_pose = ref_poses.ligands[0]
reference_pose.download()
reference_pose

In [ ]:
cd_pipeline = ConstrainedDocking(
    protein=protein,
    pocket=pocket,
    reference_ligand=reference_ligand,
    reference_pose=reference_pose,
    ligand=analogs.ligands[0],
    client=client,
)
pipeline_poses = cd_pipeline.run()
pipeline_poses.to_dataframe()

## 2. Single-ligand sync

Use an on-disk reference pose SDF directly (no prior `Docking.run()` in this cell).

In [ ]:
reference_pose_sdf = Ligand.from_sdf(BRD_DATA_DIR / "brd-2.sdf")
reference_pose_sdf.sync(
    client=client,
    remote_path=f"{REMOTE_PREFIX}/reference-pose-static.sdf",
)

cd_sync = ConstrainedDocking(
    protein=protein,
    pocket=pocket,
    reference_ligand=reference_ligand,
    reference_pose=reference_pose_sdf,
    ligand=analogs.ligands[0],
    effort=1,
    client=client,
)
sync_poses = cd_sync.run()
sync_poses

In [ ]:
sync_poses.download()
protein.show(poses=sync_poses)

## 3. Multi-ligand async

Batch constrained docking via `start()` and `watch()`.

In [ ]:
cd_async = ConstrainedDocking(
    protein=protein,
    pocket=pocket,
    reference_ligand=reference_ligand,
    reference_pose=reference_pose,
    ligands=analogs,
    client=client,
)
cd_async.start()
await cd_async.watch()
async_poses = cd_async.get_results(all_poses=True)
async_poses.to_dataframe()

## 4. MCS override

Force a common scaffold with `mcs_smarts` instead of automatic per-ligand MCS.

In [ ]:
cd_mcs = ConstrainedDocking(
    protein=protein,
    pocket=pocket,
    reference_ligand=reference_ligand,
    reference_pose=reference_pose,
    ligand=analogs.ligands[0],
    mcs_smarts="C(=O)",
    client=client,
)
mcs_poses = cd_mcs.run()
mcs_poses.to_dataframe()

## 5. Free-dock fallback

When MCS cannot match the test ligand to the reference scaffold, the tool
free-docks and sets `constrained=False` on output poses.

In [ ]:
cyclohexane = Ligand.from_smiles("C1CCCCC1")
cyclohexane.sync(
    client=client,
    remote_path=f"{REMOTE_PREFIX}/cyclohexane.sdf",
)

cd_fallback = ConstrainedDocking(
    protein=protein,
    pocket=pocket,
    reference_ligand=reference_ligand,
    reference_pose=reference_pose,
    ligand=cyclohexane,
    mcs_smarts="C(=O)",
    client=client,
)
fallback_poses = cd_fallback.run()
fallback_poses.to_dataframe()

## 6. Quote mode

Request a cost estimate without running the job.

In [ ]:
cd_quote = ConstrainedDocking(
    protein=protein,
    pocket=pocket,
    reference_ligand=reference_ligand,
    reference_pose=reference_pose,
    ligand=analogs.ligands[0],
    client=client,
)
cd_quote.run(quote=True)
cd_quote.status, cd_quote.estimate

## 7. Reference pose output

The tool echoes the reference pose used for MCS constraint derivation.

In [ ]:
reported_reference = cd_pipeline.get_reference_pose()
reported_reference.remote_path

## Visualization (optional)

Overlay reference and constrained poses in Mol*.

In [ ]:
protein.show(poses=reference_pose + pipeline_poses)